# Seen and Unseen — Experiment 3 CONFIRMATION (locked; run only after registration, the freeze and the tag)

Runs the five registered Experiment 3 configurations from the frozen git tag with the signed freeze manifest: the main confirmation (1,600 blocks; capacity 16/64/256) and the four axis configurations (latent DINOv2, latent V-JEPA 2, dose 16, dose 256; 200 blocks each). Blocks run `WORKERS` at a time; every finished block is written to a raw checkpoint on Drive, so sessions can stop and resume.

**Sealed outputs.** The raw block records (`*_raw_checkpoint.jsonl`, `*_raw.json`) carry effect values. They stay on Drive, nobody opens them, and the analyzers run only after the December 8 amendment. The notebook prints and downloads **blinded reports only** (counts, dispersions, runtime).

**Refusals built in:** the runner will not start without a freeze manifest that matches the configuration, the analyzer and the code commit, with the internal sign-offs; the clone fails if the tag does not exist.

**Runtime:** about 27 h wall-clock on one A100 with three workers, across sessions (after a disconnect: Runtime → Run all; finished blocks are skipped).

In [ ]:
#@title 0. Configuration
FREEZE_TAG = "exp3-freeze-YYYYMMDD"   #@param {type:"string"}   # the dated git tag the paper lead created at the freeze
WORKERS = 3                           #@param {type:"integer"}
RUN_MAIN = True                       #@param {type:"boolean"}
RUN_AXES = True                       #@param {type:"boolean"}
EXP3_DRIVE_DIR = "/content/drive/MyDrive/seen-and-unseen/exp3/confirmation"  #@param {type:"string"}
import os, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (switch to a GPU runtime)", "| CPUs:", os.cpu_count(), "| workers:", WORKERS)
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{EXP3_DRIVE_DIR}/SEALED_raw_records", exist_ok=True)
FREEZE = f"{EXP3_DRIVE_DIR}/freeze_manifest.json"
assert os.path.exists(FREEZE), f"put the signed freeze manifest at {FREEZE} first"
CONFIGS = {"main": "experiments/exposure2d_confirmation.json"}
if RUN_AXES:
    CONFIGS.update({"latent_dinov2": "experiments/exposure2d_confirm_latent_dinov2.json", "latent_vjepa2": "experiments/exposure2d_confirm_latent_vjepa2.json",
                    "dose16": "experiments/exposure2d_confirm_dose16.json", "dose256": "experiments/exposure2d_confirm_dose256.json"})
if not RUN_MAIN:
    CONFIGS.pop("main")
for k in CONFIGS:
    ck = f"{EXP3_DRIVE_DIR}/SEALED_raw_records/{k}_raw_checkpoint.jsonl"
    print(f"{k:14s} blocks already recorded: {sum(1 for _ in open(ck)) if os.path.exists(ck) else 0}")

In [ ]:
assert "FREEZE_TAG" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title 1. Code at the frozen tag
%cd /content
import os
if not os.path.exists("trajectory-lineage"):
    !git clone -q https://github.com/physicalreasoning/trajectory-lineage.git
%cd /content/trajectory-lineage
!git fetch -q --tags && git checkout -q {FREEZE_TAG} && git rev-parse HEAD
!pip install -q numpy h5py "transformers>=5.0" accelerate
!PYTHONPATH=src python -c "import json,sys; sys.path.insert(0,'scripts'); import freeze; print('freeze module ok')"
print("ok")

In [ ]:
assert "FREEZE_TAG" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title 2. Confirmation runs (resumable; blinded progress only)
%cd /content/trajectory-lineage
for k, cfg in CONFIGS.items():
    ck = f"{EXP3_DRIVE_DIR}/SEALED_raw_records/{k}_raw_checkpoint.jsonl"
    raw = f"{EXP3_DRIVE_DIR}/SEALED_raw_records/{k}_raw.json"
    out = f"results/{k}_confirmation_blinded.json"
    print(f"=== {k} ===")
    !PYTHONPATH=src python scripts/exposure_experiment.py --config {cfg} --out {out} --device cuda --workers {WORKERS} --freeze-manifest {FREEZE} --checkpoint {ck} --unblind {raw} 2>&1 | grep -E "^block|^resuming|refusing|Error|error"
    !cp {out} {EXP3_DRIVE_DIR}/{k}_confirmation_blinded.json

In [ ]:
assert "FREEZE_TAG" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title 3. Blinded accounting and download (no effect values)
import json, os
for k in CONFIGS:
    path = f"{EXP3_DRIVE_DIR}/{k}_confirmation_blinded.json"
    if not os.path.exists(path):
        print(f"{k:14s} not finished"); continue
    r = json.load(open(path))
    print(f"{k:14s} completed {r['blocks_completed']}/{r['blocks_scheduled']}, competent {r['blocks_base_competent']}, failures {len(r['infrastructure_failures'])}+{len(r['algorithmic_failures'])}, s/block {r['runtime']['seconds_per_block_mean']:.0f}")
!cd {EXP3_DRIVE_DIR} && zip -q -j /content/exp3_confirmation_blinded.zip *_confirmation_blinded.json
!ls -la /content/exp3_confirmation_blinded.zip
from google.colab import files
files.download("/content/exp3_confirmation_blinded.zip")

### After all five report complete
The analysis is a separate, later step after the December 8 amendment: `scripts/analyze_2d.py` on the main raw export and `scripts/analyze_exp3_axes.py` across all five, both of which refuse to run without the freeze manifest. Do not open `SEALED_raw_records/`.